In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
import dtale
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi 
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance 


from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
import matplotlib.pyplot as plt

import gc




In [2]:
cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")

cleaned_application_train["ratio_debt_income"] = (cleaned_application_train["amt_credit"] /  (cleaned_application_train["amt_income_total"]))

cleaned_application_train["ratio_debt_age"]= cleaned_application_train["amt_credit"] /(cleaned_application_train["days_birth"] * -1) 
cleaned_application_train["ratio_days_employed_days_lived"]= cleaned_application_train["days_employed"] /(cleaned_application_train["days_birth"] * -1) 
cleaned_application_train["kui_ratio"] =  np.where(cleaned_application_train["days_employed"] != 0, cleaned_application_train["amt_credit"] / ((cleaned_application_train["days_employed"] * -1) * cleaned_application_train["amt_income_total"]), 0)
cleaned_application_train["ratio_good_credit"]= cleaned_application_train["amt_goods_price"] / cleaned_application_train["amt_credit"]
cleaned_application_train["ratio_annuity_income"] = cleaned_application_train["amt_annuity"] / cleaned_application_train["amt_income_total"]
#cleaned_application_train["toxic_feature_1"] = cleaned_application_train["cnt_children"] / cleaned_application_train["amt_income_total"]

cleaned_application_train["credit_duration"]= cleaned_application_train["amt_credit"] / cleaned_application_train["amt_annuity"]
cleaned_application_train["ext_1_x_2"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_2"]
cleaned_application_train["ext_2_x_3"] = cleaned_application_train["ext_source_2"] * cleaned_application_train["ext_source_3"]
cleaned_application_train["ext_1_x_3"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_3"]








#cleaned_application_train = pd.get_dummies(cleaned_application_train,columns=["occupation_type"])


#cleaned_application_train["amount_of_scores_in_missing"] = cleaned_application_train["ext_source_1_is_missing"] + cleaned_application_train["ext_source_2_is_missing"] + cleaned_application_train["ext_source_3_is_missing"]
ext_cols = ['ext_source_1', 'ext_source_2', 'ext_source_3']

building_features_names = [
    "APARTMENTS_AVG","BASEMENTAREA_AVG","YEARS_BEGINEXPLUATATION_AVG","YEARS_BUILD_AVG",
    "COMMONAREA_AVG","ELEVATORS_AVG","ENTRANCES_AVG","FLOORSMAX_AVG","FLOORSMIN_AVG",
    "LANDAREA_AVG","LIVINGAPARTMENTS_AVG","LIVINGAREA_AVG","NONLIVINGAPARTMENTS_AVG",
    "NONLIVINGAREA_AVG","APARTMENTS_MODE","BASEMENTAREA_MODE","YEARS_BEGINEXPLUATATION_MODE",
    "YEARS_BUILD_MODE","COMMONAREA_MODE","ELEVATORS_MODE","ENTRANCES_MODE","FLOORSMAX_MODE",
    "FLOORSMIN_MODE","LANDAREA_MODE","LIVINGAPARTMENTS_MODE","LIVINGAREA_MODE",
    "NONLIVINGAPARTMENTS_MODE","NONLIVINGAREA_MODE","APARTMENTS_MEDI","BASEMENTAREA_MEDI",
    "YEARS_BEGINEXPLUATATION_MEDI","YEARS_BUILD_MEDI","COMMONAREA_MEDI","ELEVATORS_MEDI",
    "ENTRANCES_MEDI","FLOORSMAX_MEDI","FLOORSMIN_MEDI","LANDAREA_MEDI","LIVINGAPARTMENTS_MEDI",
    "LIVINGAREA_MEDI","NONLIVINGAPARTMENTS_MEDI","NONLIVINGAREA_MEDI","FONDKAPREMONT_MODE",
    "HOUSETYPE_MODE","TOTALAREA_MODE","WALLSMATERIAL_MODE","EMERGENCYSTATE_MODE"
    ]

building_features_names = [col.lower() for col in building_features_names]

categorical_bldg = ['fondkapremont_mode', 'housetype_mode', 'wallsmaterial_mode', 'emergencystate_mode']

numeric_bldg = [col for col in building_features_names if col not in categorical_bldg]



# Agregaciones horizontales (axis=1)
cleaned_application_train["ext_source_mean"] = cleaned_application_train[ext_cols].mean(axis=1)
#cleaned_application_train["ext_source_max"] = cleaned_application_train[ext_cols].max(axis=1)
#cleaned_application_train["ext_source_min"] = cleaned_application_train[ext_cols].min(axis=1)
cleaned_application_train["ext_source_std"] = cleaned_application_train[ext_cols].std(axis=1)




# building_agg
cleaned_application_train["building_score_mean"] = cleaned_application_train[numeric_bldg].mean(axis=1)
cleaned_application_train["building_score_max"] = cleaned_application_train[numeric_bldg].max(axis=1)
cleaned_application_train["building_score_min"] = cleaned_application_train[numeric_bldg].min(axis=1)
cleaned_application_train["building_score_std"] = cleaned_application_train[numeric_bldg].std(axis=1)
cleaned_application_train["building_score_sum"] = cleaned_application_train[numeric_bldg].sum(axis=1)

cleaned_application_train["building_features_nan_count"] = cleaned_application_train[building_features_names].isnull().sum(axis=1)

cleaned_application_train= cleaned_application_train.drop(columns=numeric_bldg)
#cleaned_application_train= cleaned_application_train.drop(columns=["flag_city_not_work"] )

cleaned_application_train.to_parquet(cfg.PROCESSED_DIR / "application_train_for_experiments")

